# Alzheimer's Dataset: Analysis Roadmap, recommendations and notes

## About the Data

The NACC dataset (`investigator_nacc72.csv`) is a large longitudinal file (500+ MB) tracking thousands of patients over multiple years. It covers demographics, medical history, medications, and cognitive test scores. A companion file (`rdd-gen.csv`) contains genetic data, specifically the APOE e4 allele, the strongest known genetic risk factor for Alzheimer's.

The goals are to identify what causes or accelerates Alzheimer's (risk factors) and to find everyday, non-invasive actions that protect the brain or slow decline (protective factors).

## Pick a Research Question

- **Preventing onset:** Start with patients who entered the study as Cognitively Normal. Track how many converted to MCI or Dementia, and how long it took.
- **Slowing progression:** Start with patients already diagnosed with MCI or early-stage Alzheimer's. Measure the rate of cognitive decline over visits.

## Mandatory Controls

Every model must account for these biological confounders:

| Variable | Meaning |
|----------|---------|
| `NACCAGE` | Age at visit, the single biggest risk factor |
| `SEX` | Biological sex |
| `EDUC` | Years of education, builds cognitive reserve |
| `APOE4` (`NACCNE4S`) | Number of high-risk e4 alleles (0, 1, or 2) |

## Modifiable Factors to Investigate

These are the variables most likely to reveal actionable insights:

**Heart health** - Blood pressure (`HYPERTEN`), diabetes (`DIABETES`), cholesterol (`HYPERCHO`), BMI (`NACCBMI`). What helps the heart tends to help the brain.

**Mental health and sleep** - Depression history (`DEP`), sleep apnea (`SLEEPAP`). Poor sleep and untreated depression are both linked to faster decline.

**Habits** - Smoking (`TOBAC30`), alcohol abuse (`ALCOHOL`).

**Medications** - Statins, antihypertensives, and diabetes drugs like Metformin are of particular interest. Some research suggests they may offer incidental brain protection.

## Recommended Methods

This is observational data, so correlation alone is not sufficient. Use these approaches:

- **Machine Learning (feature discovery)**  Train a Random Forest or XGBoost model to predict cognitive decline. Use SHAP values to visualize which modifiable factors matter most. See `ml_feature_importance.ipynb`.
- **Survival Analysis (time-to-event)**  Cox Proportional Hazards models to compare how long different groups (zB. smokers vs. non-smokers) stay dementia-free, controlling for age and genetics. Library: `lifelines`.
- **Propensity Score Matching (causal inference)**  Match two nearly identical patients where the only difference is a specific treatment or habit. Compare their cognitive trajectories. Libraries: `DoWhy`, `EconML`.

# Post frequency table implementation:
### I got most of the info by looking at the ML's and analysis' that we have at this point (post frequency table). ML methods listed below were just me looking into various models and their strength+features and noting them down with a possible next step ***https://www.geeksforgeeks.org/machine-learning/machine-learning-models/***


>A. Predict the "Transition" (The "Who will decline?" Model)
Currently, the ML models predict if a patient is impaired at baseline. It is much more clinically valuable to predict who will move from "Normal to MCI" or "MCI to AD", perhaps even backwards.

*Using "Forward" group transition labels as the target variable for your XGBoost model.
Question: Can baseline lifestyle factors predict the speed or likelihood of future decline?*

>B. APOE4 Interactions. The APOE4 allele is the strongest genetic risk factor, but its impact often interacts with lifestyle.

*Perform a segmented SHAP analysis. Compare feature importance for APOE4 carriers vs. non-carriers.
Question: Does Hypertension, BMI or ***depression*** (we saw that depression is one, if not the leading factor for cogn. imp.) have a stronger negative impact on someone who is already genetically predisposed?*

>C. Cluster Analysis (Identifying "Patient Phenotypes")
Not all Alzheimer's progression looks the same.

*Using unsupervised learning (K-Means or HDBSCAN) on the "Progression" groups.
Question: Are there distinct "subtypes" of patients? (zB. a "Cardiovascular subtype" vs. a "Depressive/Mental Health subtype").*

>D. Time-to-Event
Instead of a binary "Yes/No" for impairment, model the time until a transition occurs.

*Using a Cox Proportional Hazards model or Random Survival Forests.
Question: Which factors most significantly delay the onset of symptoms (extending the "Stable" period)?*